# 02. CRNN ??? OCR

????? 3-4: OCR-???????, ?????? CRNN, ???????????? ? ????????? ??????.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_ROOT = ROOT / 'data' / 'raw' / 'autoriaNumberplateOcrRu'
OUT_ROOT = ROOT / 'outputs' / 'crnn'
RUN_CRNN_TRAINING = False

ALPHABET = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
BLANK_IDX = 0
CHAR_TO_IDX = {ch: i + 1 for i, ch in enumerate(ALPHABET)}
IDX_TO_CHAR = {i + 1: ch for i, ch in enumerate(ALPHABET)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
print('DATA_ROOT:', DATA_ROOT)

DEVICE: cpu
DATA_ROOT: C:\Users\User\Documents\gen-img\designing_neural_network_architectures_2025_02\seminar_01\data\raw\autoriaNumberplateOcrRu


## 1. OCR-???????

????? ????? ? `split/img`, ????????? ? `split/ann`, ????? ?????? ??????? ?? ???? `description`.

In [2]:
def read_annotations(root: Path, split: str):
    rows = []
    ann_dir = root / split / 'ann'
    img_dir = root / split / 'img'
    for ann_path in sorted(ann_dir.glob('*.json')):
        item = json.loads(ann_path.read_text(encoding='utf-8'))
        label = ''.join(ch for ch in str(item.get('description', '')).upper() if ch in CHAR_TO_IDX)
        name = str(item.get('name') or ann_path.stem)
        img_path = img_dir / f'{name}.png'
        if label and img_path.exists():
            rows.append({'image': img_path, 'label': label, 'name': name})
    return rows

counts = []
for split in ['train', 'val', 'test']:
    rows = read_annotations(DATA_ROOT, split)
    counts.append({'split': split, 'available': len(rows)})
display(pd.DataFrame(counts))

train_rows = read_annotations(DATA_ROOT, 'train')[:8000]
val_rows = read_annotations(DATA_ROOT, 'val')[:1200]
test_rows = read_annotations(DATA_ROOT, 'test')[:1200]
display(pd.DataFrame([
    {'split': 'train_used', 'count': len(train_rows)},
    {'split': 'val_used', 'count': len(val_rows)},
    {'split': 'test_used', 'count': len(test_rows)},
]))

,split,available
0,train,49382
1,val,4893
2,test,2845


,split,count
0,train_used,8000
1,val_used,1200
2,test_used,1200


## 2. Dataset, collate, CRNN

In [3]:
def pil_to_tensor(img: Image.Image, width=128, height=32):
    img = img.convert('L').resize((width, height), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)

class PlateDataset(Dataset):
    def __init__(self, rows, width=128, height=32):
        self.rows = rows
        self.width = width
        self.height = height
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        with Image.open(row['image']) as img:
            tensor = pil_to_tensor(img, self.width, self.height)
        target = torch.tensor([CHAR_TO_IDX[ch] for ch in row['label']], dtype=torch.long)
        return tensor, target, row['label'], row['name']

def collate_batch(batch):
    images, targets, labels, names = zip(*batch)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    return torch.stack(images), torch.cat(targets), target_lengths, list(labels), list(names)

class CRNN(nn.Module):
    def __init__(self, hidden=64, vocab_size=len(ALPHABET) + 1, lstm_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, hidden * 2, 3, padding=1), nn.BatchNorm2d(hidden * 2), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(hidden * 2, hidden * 2, 3, padding=1), nn.BatchNorm2d(hidden * 2), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(hidden * 2, hidden * 2, kernel_size=(2, 1)), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(hidden * 2, hidden, num_layers=lstm_layers, bidirectional=True, dropout=0.1 if lstm_layers > 1 else 0)
        self.fc = nn.Linear(hidden * 2, vocab_size)
    def forward(self, images):
        feats = self.cnn(images)
        feats = feats.squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(feats)
        return self.fc(seq).log_softmax(2)

def decode_batch(log_probs):
    pred_ids = log_probs.argmax(2).permute(1, 0).detach().cpu().numpy()
    decoded = []
    for seq in pred_ids:
        chars = []
        prev = None
        for idx in seq:
            idx = int(idx)
            if idx != BLANK_IDX and idx != prev:
                chars.append(IDX_TO_CHAR.get(idx, ''))
            prev = idx
        decoded.append(''.join(chars))
    return decoded

print(CRNN())

CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(128, eps=1e-05, momentum=0.1,

## 3. ????????????

????????? ?????? ?????????? ?????? `RUN_CRNN_TRAINING = True`. ???? ???????????? ??????????? ???????.

In [4]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for images, targets, target_lengths, labels, names in loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        target_lengths = target_lengths.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        log_probs = model(images)
        input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long, device=DEVICE)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += float(loss.item())
    return total / max(len(loader), 1)

if RUN_CRNN_TRAINING:
    cfg = {'width': 128, 'height': 32, 'hidden': 64, 'batch_size': 64, 'lr': 5e-4, 'epochs': 20}
    train_loader = DataLoader(PlateDataset(train_rows, cfg['width'], cfg['height']), batch_size=cfg['batch_size'], shuffle=True, collate_fn=collate_batch)
    model = CRNN(hidden=cfg['hidden']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
    for epoch in range(cfg['epochs']):
        loss = train_one_epoch(model, train_loader, optimizer, criterion)
        print(epoch + 1, loss)
else:
    print('???????? ??? ?????????. ??????? ???????? ?? outputs/crnn/*.json')

???????? ??? ?????????. ??????? ???????? ?? outputs/crnn/*.json


In [5]:
summary = json.loads((OUT_ROOT / 'experiments_summary.json').read_text(encoding='utf-8'))
rows = []
for item in summary:
    cfg = item['config']
    rows.append({
        'experiment': item['name'],
        'input': f"{cfg['height']}x{cfg['width']}",
        'hidden': cfg['hidden'],
        'augment': cfg['augment'],
        'scheduler': cfg['scheduler'],
        'test_CER': item['test_cer'],
        'test_exact': item['test_exact_match'],
        'val_CER': item['val_cer'],
    })
display(pd.DataFrame(rows))

final = json.loads((OUT_ROOT / 'final_metrics.json').read_text(encoding='utf-8'))
final_metrics = final['metrics']
display(pd.DataFrame([
    {'split': 'val', **final_metrics['val']},
    {'split': 'test', **final_metrics['test']},
]))
print('Final checkpoint:', ROOT / final['final_checkpoint'])
print('Best final checkpoint:', ROOT / final['best_final_checkpoint'])

,experiment,input,hidden,augment,scheduler,test_CER,test_exact,val_CER
0,baseline_32x128,32x128,64,False,none,0.924160,0.0,0.920969
1,wide_32x160,32x160,96,False,none,0.911074,0.0,0.909496
2,augmented_32x128,32x128,64,True,none,0.883117,0.0,0.882394
3,augmented_cosine_32x160,32x160,96,True,cosine,0.969367,0.0,0.968051


,split,loss,cer,exact_match
0,val,0.018759,0.003165,0.978333
1,test,0.032405,0.005353,0.965000


Final checkpoint: C:\Users\User\Documents\gen-img\designing_neural_network_architectures_2025_02\seminar_01\outputs\crnn\final_crnn.pt
Best final checkpoint: C:\Users\User\Documents\gen-img\designing_neural_network_architectures_2025_02\seminar_01\outputs\crnn\final_long\best_crnn.pt


## 4. ??????? OCR

In [6]:
checkpoint = torch.load(OUT_ROOT / 'final_crnn.pt', map_location=DEVICE)
cfg = checkpoint['config']
model = CRNN(hidden=cfg['hidden'], lstm_layers=cfg['lstm_layers']).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

sample_ds = PlateDataset(test_rows[:12], width=cfg['width'], height=cfg['height'])
sample_loader = DataLoader(sample_ds, batch_size=12, shuffle=False, collate_fn=collate_batch)
images, targets, target_lengths, labels, names = next(iter(sample_loader))
with torch.no_grad():
    preds = decode_batch(model(images.to(DEVICE)))

display(pd.DataFrame({'name': names, 'GT': labels, 'PRED': preds, 'ok': [g == p for g, p in zip(labels, preds)]}))

,name,GT,PRED,ok
0,A001BP54,A001BP54,A001BP54,True
1,A001PC71,A001PC71,A001PC71,True
2,A002KX152,A002KX152,A002KX152,True
3,A002XC763,A002XC763,O002XC763,False
4,A002XY89,A002XY89,A002XY89,True
5,A003CX196,A003CX196,A003CX196,True
6,A004OE23,A004OE23,A004OE23,True
7,A005AX26,A005AX26,A005AX26,True
8,A006AA10,A006AA10,A006AA10,True
9,A007AE799,A007AE799,A007AE799,True
